In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import sys
import sys; sys.path.append("/data/jerrylee/pjt/BIGFAM.v.0.1")

from BIGFAM import obj2, tools
import importlib

In [2]:
source = "GS"

# Step 1. Load data

In [3]:
# load FR-reg
frreg_path = f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/{source}/frreg/REL"
fns = os.listdir(frreg_path)
len(fns)

32

# Step 2. Estimate X

In [13]:
df = pd.DataFrame(
    columns=[
        "pheno", "lambda", "weight", 
        "Vx", "lower_Vx", "upper_Vx"
    ]
)

for ii, fn in enumerate(tqdm(fns)):
    pheno = fn.split(".")[0]
    print(f"[{ii+1}/{len(fns)}] {pheno}")
    
    # load FR-reg results
    df_frreg = pd.read_csv(f"{frreg_path}/{fn}", 
                           delim_whitespace=True)
    try:
        for weight in range(-2, 9):
            df_x = obj2.estimateX(
                df_frreg,
                n_resample=100,
                regout_bin=["DOR"],
                alpha_dicts={"type": "lambda", "weight":weight},)
        
            df.loc[len(df)] = [
                pheno, df_x["lambda"].mean(), weight,
                np.median(df_x["X"]), np.percentile(df_x["X"], 2.5), 
                np.percentile(df_x["X"], 97.5)
            ]
    except:
        print(fn)
    break

  0%|          | 0/32 [00:00<?, ?it/s]

[1/32] avg_dia


  0%|          | 0/32 [00:25<?, ?it/s]


In [17]:
df_x

,lambda,alpha,X
0,0.271901,33474.813382,1.219913e-06
1,0.271901,33474.813382,1.767795e-06
2,0.271901,33474.813382,2.073420e-06
3,0.271901,33474.813382,9.366114e-07
4,0.271901,33474.813382,1.231018e-06
...,...,...,...
95,0.271901,33474.813382,2.093614e-06
96,0.271901,33474.813382,3.116449e-06
97,0.271901,33474.813382,2.411753e-06
98,0.271901,33474.813382,3.098087e-07


In [16]:
df

,pheno,lambda,weight,Vx,lower_Vx,upper_Vx
0,avg_dia,0.271901,-2,0.090814,3.309034e-02,0.154682
1,avg_dia,0.271901,-1,0.068248,1.948109e-02,0.117571
2,avg_dia,0.271901,0,0.039943,1.561008e-02,0.068097
3,avg_dia,0.271901,1,0.013450,4.114673e-03,0.021867
4,avg_dia,0.271901,2,0.004053,1.588569e-03,0.007054
5,avg_dia,0.271901,3,0.001244,4.377091e-04,0.002098
6,avg_dia,0.271901,4,0.000334,1.305468e-04,0.000540
7,avg_dia,0.271901,5,0.000095,1.774118e-05,0.000144
8,avg_dia,0.271901,6,0.000027,1.127789e-05,0.000044
9,avg_dia,0.271901,7,0.000007,1.961602e-06,0.000011


# Step 3. Estimate XmXf

In [4]:
df = pd.DataFrame(
    columns=[
        "pheno", "lambda", "weight", 
        "Vx_male", "lower_Vx_male", "upper_Vx_male",
        "Vx_female", "lower_Vx_female", "upper_Vx_female",
        "r", "lower_r", "upper_r",
    ]
)

for ii, fn in enumerate(tqdm(fns)):
    pheno = fn.split(".")[0]
    print(f"[{ii+1}/{len(fns)}] {pheno}")
    
    # load FR-reg results
    df_frreg = pd.read_csv(f"{frreg_path}/{fn}", 
                           delim_whitespace=True)
    try:
        for weight in range(-2, 9):
            df_x = obj2.estimateXmXfR(
                df_frreg,
                n_resample=100,
                regout_bin=["DOR", "sex_type"],
                alpha_dicts={"type": "lambda", "weight":weight},)
            
            df.loc[len(df)] = [
                pheno, df_x["lambda"].mean(), weight,
                np.median(df_x["Xmale"]), np.percentile(df_x["Xmale"], 2.5), 
                np.percentile(df_x["Xmale"], 97.5),
                np.median(df_x["Xfemale"]), np.percentile(df_x["Xfemale"], 2.5), 
                np.percentile(df_x["Xfemale"], 97.5),
                np.median(df_x["r"]), np.percentile(df_x["r"], 2.5), 
                np.percentile(df_x["r"], 97.5),
            ]
    except:
        print(fn)

  0%|          | 0/32 [00:00<?, ?it/s]

[1/32] avg_dia


  0%|          | 0/32 [01:01<?, ?it/s]


In [9]:
df

,pheno,lambda,weight,Vx_male,lower_Vx_male,upper_Vx_male,Vx_female,lower_Vx_female,upper_Vx_female,r,lower_r,upper_r
0,avg_dia,0.271901,-2,0.294085,0.210913,0.351298,0.106977,0.032429,0.203574,-0.2,-1.0,0.6
